# Day 22 — DPO on Colab Pro A100 40 GB

This notebook runs the cost-aware workflow in `configs/colab_a100.yaml`: GPU fail-fast, baseline evaluation, LoRA DPO, post-training evaluation, regression prompts, and adapter-only export. Run cells from top to bottom.

In [ ]:
# Fail before downloads if Colab did not assign the requested GPU.
import subprocess
import torch

assert torch.cuda.is_available(), 'CUDA is unavailable — stop this runtime.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GiB')
assert 'A100' in gpu_name and vram_gb >= 38, (
    f'Expected A100 40 GB, received {gpu_name} with {vram_gb:.1f} GiB. '
    'Stop the runtime now to avoid wasting compute units.'
)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Load credentials from Colab Secrets. Never paste token values into this notebook.
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
xah_api_key = userdata.get('XAH_API_KEY')
assert hf_token, 'Add HF_TOKEN to Colab Secrets and enable notebook access.'
assert xah_api_key, 'Add XAH_API_KEY to Colab Secrets and enable notebook access.'
os.environ['HF_TOKEN'] = hf_token
os.environ['OPENAI_API_KEY'] = xah_api_key
os.environ['XAH_API_KEY'] = xah_api_key
os.environ['OPENAI_BASE_URL'] = 'https://api.xah.io/v1'
os.environ['OPENAI_MODEL'] = 'levuphong2909/gpt-5.6-luna'
print('HF and XAH credentials loaded from Colab Secrets (values hidden).')
del hf_token, xah_api_key

In [ ]:
# Clone the submitted repository. Push local changes before running this cell.
import os
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/hardy410/K3-Track3-Day22-2A202601790-NguyenDinhLienThanh.git'
REPO_DIR = Path('/content/K3-Track3-Day22-2A202601790-NguyenDinhLienThanh')
BRANCH = 'main'

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
# Pin the tested training stack; keep Colab's CUDA-enabled PyTorch wheel.
import subprocess
import sys

# Colab currently preinstalls torchao 0.10, which PEFT rejects. This lab does not use it.
torchao_cleanup = subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
    capture_output=True,
    text=True,
)
print(torchao_cleanup.stdout.strip() or 'torchao was not installed')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==5.15.1',
        'datasets==5.0.1',
        'trl==1.10.0',
        'peft==0.20.0',
        'accelerate>=1.10,<2',
        'pytest>=8,<10',
        'tqdm>=4.66,<5',
    ],
    check=True,
)
print('Training dependencies installed without replacing torch:', torch.__version__)

In [ ]:
# Fast preflight: code quality, unit tests, package versions, and YAML parsing.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
pip_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'],
    capture_output=True,
    text=True,
)
if pip_check.returncode:
    print('WARNING: Colab has preinstalled package conflicts; continuing after core imports.')
    print(pip_check.stdout or pip_check.stderr)
else:
    print('pip check: no broken requirements')
from datasets import __version__ as datasets_version
from peft import __version__ as peft_version
from transformers import __version__ as transformers_version
from trl import __version__ as trl_version
print({
    'torch': torch.__version__,
    'transformers': transformers_version,
    'datasets': datasets_version,
    'trl': trl_version,
    'peft': peft_version,
})

In [ ]:
# Mount Drive before training so final artifacts can be copied immediately.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/VinUni/Day22-DPO')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Artifacts will be copied under: {DRIVE_ROOT}')

In [ ]:
# Baseline evaluation -> LoRA DPO -> post evaluation -> adapter/metrics export.
import subprocess
import sys

command = [
    sys.executable,
    '-u',
    'scripts/train_dpo_colab.py',
    '--config',
    'configs/colab_a100.yaml',
]
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'DPO workflow failed with exit code {return_code}. '
        'The full child-process traceback is printed immediately above.'
    )

In [ ]:
# Copy the compact adapter and evidence to a unique Drive directory.
from datetime import datetime, timezone
import json
import shutil
from google.colab import files

LOCAL_OUTPUT = Path('outputs/colab-a100-dpo')
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_OUTPUT = DRIVE_ROOT / run_id
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUTPUT)
metrics = json.loads((LOCAL_OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(f'Persistent copy: {DRIVE_OUTPUT}')
zip_base = Path('/content') / f'day22-dpo-results-{run_id}'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=LOCAL_OUTPUT)
print(f'Downloading ZIP: {zip_path}')
files.download(zip_path)

## Stop the runtime

After confirming the Drive copy contains `metrics.json` and `final-adapter/`, use **Runtime → Disconnect and delete runtime** to stop compute-unit consumption.